# Weather features joined to AQI model examples

This walkthrough builds the same-day station summaries and joins them to the same station's AQI category on the exact next calendar day. Feature engineering lives in `weather_model_comparison.py` so this notebook and the repeatable model comparison use the same implementation.

Run from the repository root. Weather readings are observed values from the feature date, available after that day's hourly observations are complete.

In [ ]:
from pathlib import Path
import pandas as pd
from weather_model_comparison import SOURCE, POLLUTANTS, WEATHER, build_supervised_dataset, daily_weather


In [ ]:
hourly = pd.read_csv(SOURCE)
print(f'{len(hourly):,} hourly rows; {hourly.station.nunique()} stations')
print(hourly.columns.tolist())


## Daily weather summaries

Numeric features are daily means except rainfall, which is summed; temperature also has a daily range. Wind direction uses circular sine/cosine means. Each summary requires at least 20 valid hourly readings.

In [ ]:
weather = daily_weather(hourly)
print('Station-days:', f'{len(weather):,}')
print('Complete weather rows:', f'{weather[WEATHER].notna().all(axis=1).sum():,}')
display(weather.head())
display(weather[WEATHER].isna().sum().rename('missing station-days').to_frame())


## Join features to next-day targets

The function joins pollutant and weather summaries by station and feature date. It moves each valid AQI label back exactly one calendar day before joining, so missing target dates do not get skipped. The joined rows are the examples used by the temporal comparison.

In [ ]:
examples = build_supervised_dataset(hourly)
print(f'{len(examples):,} supervised station-days')
print('Date range:', examples.date.min().date(), 'to', examples.date.max().date())
print('Duplicate station-date examples:', examples.duplicated(['station', 'date']).sum())
display(examples[['date', 'station', *POLLUTANTS, *WEATHER, 'target_category', 'target_AQI', 'target_main_pollutant']].head(10))


In [ ]:
# Optional export matching the dataset artifact produced by weather_model_comparison.py
output_path = Path('weather_augmented_model_dataset.csv')
examples[['date', 'station', *POLLUTANTS, *WEATHER, 'target_category', 'target_AQI', 'target_main_pollutant']].to_csv(output_path, index=False)
print(f'Saved {output_path.resolve()}')


## Matched time-series tuning

This section reserves the latest 20% of dates as the final holdout. It tunes both feature sets on the earlier dates using the same four expanding-window folds, macro-F1 score, Random Forest parameter grid, median imputation, and class weighting. The held-out data is not used to select parameters.

In [ ]:
from weather_model_tuning import tune_both
cv_results, fitted_models, train, holdout = tune_both(examples)
display(cv_results.sort_values(['feature_set', 'mean_cv_macro_f1'], ascending=[True, False]))


In [ ]:
cv_output = Path('weather_model_tuning_results.csv')
cv_results.to_csv(cv_output, index=False)
print(f'Saved candidate CV scores to {cv_output.resolve()}')


## Held-out evaluation

Both best estimators from the matched search are refit on the full training period by GridSearchCV. This section compares them on the same untouched latest 20% of dates, reporting overall metrics, per-category recall, and confusion matrices.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from aqi_labels import AQI_CATEGORIES
labels = list(AQI_CATEGORIES.values())
holdout_rows, confusion_tables = [], {}
for name, model in fitted_models.items():
    features = POLLUTANTS if name == 'pollutant_only' else POLLUTANTS + WEATHER
    y_true = holdout.target_category
    y_pred = model.predict(holdout[features])
    holdout_rows.append({'feature_set': name, 'accuracy': accuracy_score(y_true, y_pred), 'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0), 'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)})
    display(pd.DataFrame(classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)).T.loc[labels])
    confusion_tables[name] = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels)
    display(confusion_tables[name])
holdout_metrics = pd.DataFrame(holdout_rows)
display(holdout_metrics)


In [ ]:
holdout_metrics.to_csv('weather_model_holdout_metrics.csv', index=False)
print('Saved weather_model_holdout_metrics.csv')


## Stability by time block, station, and season

The untouched holdout is split into two consecutive blocks. The same models are evaluated by station and season to show where gains vary. These are subgroup checks within one historical holdout, not independent replications.

In [ ]:
from weather_model_stability import main as run_stability
run_stability()
periods = pd.read_csv('weather_model_stability_periods.csv')
stations = pd.read_csv('weather_model_stability_stations.csv')
seasons = pd.read_csv('weather_model_stability_seasons.csv')
display(periods.pivot(index='period', columns='feature_set', values=['accuracy', 'macro_f1']))
display(stations.pivot(index='group', columns='feature_set', values=['accuracy', 'macro_f1']))
display(seasons.pivot(index='group', columns='feature_set', values=['accuracy', 'macro_f1']))


## Train the weather-enabled app artifact

After reviewing the candidate and its held-out evaluation, this optional final section refits the category, numeric AQI, and main-pollutant models on all eligible station-days and writes the app artifact and dataset. It also records the holdout metrics from the evaluation stage.

In [ ]:
from train_weather_model_release import main as train_weather_release
train_weather_release()
